# Feature Engineering

Step-by-step feature extraction for the ML pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

In [ ]:
import os

data_path = os.path.join("..", "..", "data_pipeline", "data", "students.json")
with open(data_path, "r") as f:
    students_raw = json.load(f)

df = pd.json_normalize(students_raw)
print(f"Loaded {len(df)} students")
df.head(2)

## 1. Academic Features

In [ ]:
features = pd.DataFrame(index=range(len(df)))

features["cgpa_norm"] = df["cgpa"].values / 10.0
features["year_norm"] = df["year"].values / 4.0

pace_map = {"slow": 0.0, "moderate": 0.5, "fast": 1.0}
features["learning_pace_encoded"] = df["learning_pace"].map(pace_map).values

dept_dummies = pd.get_dummies(df["department"], prefix="dept").astype(float)
for col in dept_dummies.columns:
    features[col] = dept_dummies[col].values

print("Academic features:")
print(features.head())
print(f"\nShape so far: {features.shape}")

## 2. Skill Features

In [ ]:
subjects = [
    "DSA", "OOP", "DBMS", "OS", "Networks",
    "Mathematics", "Physics", "English", "Machine Learning", "Web Development"
]

self_ratings = np.zeros((len(students_raw), len(subjects)))
peer_ratings = np.zeros((len(students_raw), len(subjects)))
grade_points = np.zeros((len(students_raw), len(subjects)))

for i, student in enumerate(students_raw):
    skill_lookup = {s["subject"]: s for s in student["skills"]}
    for j, subj in enumerate(subjects):
        if subj in skill_lookup:
            self_ratings[i, j] = skill_lookup[subj]["self_rating"] / 10.0
            peer_ratings[i, j] = skill_lookup[subj]["peer_rating"] / 10.0
            grade_points[i, j] = skill_lookup[subj]["grade_points"] / 10.0

for j, subj in enumerate(subjects):
    short = subj.lower().replace(" ", "_")
    features[f"skill_self_{short}"] = self_ratings[:, j]
    features[f"skill_peer_{short}"] = peer_ratings[:, j]
    features[f"skill_grade_{short}"] = grade_points[:, j]

all_skill_vals = (self_ratings + peer_ratings + grade_points) / 3.0
features["mean_skill"] = all_skill_vals.mean(axis=1)
features["std_skill"] = all_skill_vals.std(axis=1)
features["min_skill"] = all_skill_vals.min(axis=1)
features["max_skill"] = all_skill_vals.max(axis=1)
features["skill_range"] = features["max_skill"] - features["min_skill"]

print(f"After skill features: {features.shape}")
print(features[["mean_skill", "std_skill", "min_skill", "max_skill", "skill_range"]].describe())

## 3. Availability Bitmap

In [ ]:
days = ["mon", "tue", "wed", "thu", "fri", "sat", "sun"]
slots = ["morning", "afternoon", "evening"]

avail_bitmap = np.zeros((len(students_raw), 21))

for i, student in enumerate(students_raw):
    for entry in student["availability"]:
        day_idx = entry["day_of_week"]
        slot_idx = slots.index(entry["slot"])
        bit_pos = day_idx * 3 + slot_idx
        if entry["is_available"]:
            avail_bitmap[i, bit_pos] = 1.0

for d_idx, day in enumerate(days):
    for s_idx, slot in enumerate(slots):
        col_name = f"avail_{day}_{slot}"
        features[col_name] = avail_bitmap[:, d_idx * 3 + s_idx]

features["total_available_slots"] = avail_bitmap.sum(axis=1)

print(f"After availability features: {features.shape}")
print(f"\nAvailable slots stats:")
print(features["total_available_slots"].describe())

## 4. Feature Correlation Matrix

In [ ]:
key_features = [
    "cgpa_norm", "year_norm", "learning_pace_encoded",
    "mean_skill", "std_skill", "skill_range",
    "total_available_slots",
]

fig, ax = plt.subplots(figsize=(10, 8))
corr = features[key_features].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Feature Correlation Matrix (Key Features)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Feature Distributions

In [ ]:
plot_features = [
    "cgpa_norm", "year_norm", "learning_pace_encoded",
    "mean_skill", "std_skill", "total_available_slots",
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, feat in enumerate(plot_features):
    axes[idx].hist(features[feat], bins=25, edgecolor="black", alpha=0.7, color=sns.color_palette()[idx])
    axes[idx].set_title(feat, fontsize=12)
    axes[idx].set_xlabel("Value")
    axes[idx].set_ylabel("Count")
    axes[idx].axvline(features[feat].mean(), color="red", linestyle="--", linewidth=1.5)

fig.suptitle("Key Feature Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Summary

Total features extracted: 45+ per student

| Category | Count | Description |
|---|---|---|
| Academic | 6 | CGPA, year, pace, department one-hot |
| Skill | 35 | 10 subjects × 3 ratings + 5 aggregates |
| Availability | 22 | 21-dim bitmap + total count |
| **Total** | **63** | Full feature vector per student |